# SaProt -- Geometric Stability Scaling Experiment

Tests geometric stability across **3 sizes** of SaProt, a structure-aware
protein language model.

## Why This Matters

- **SaProt** extends ESM with 3Di structural alphabet tokens from Foldseek
- 3 sizes: 35M, 650M, 1.3B -- enables scaling analysis
- Same ESM architecture but with structure-awareness -- tests whether
  structural information affects embedding geometry stability
- Direct comparison to ESM-2 (sequence-only) at matching sizes

## Notes on Structure Tokens

SaProt was trained with interleaved AA + 3Di tokens (from Foldseek).
Without structure predictions, we feed **sequence-only** input, which
the tokenizer handles natively. This tests the model's sequence-only
geometric stability.

## Setup

1. Upload `evaluation_harness.py` and `perturbation_protocol.py`
2. Change Runtime to GPU (A100 recommended for 1.3B model)
3. Run cells in order

---

In [ ]:
# Install Dependencies

print("Installing core dependencies...")
!pip install -q torch transformers shesha-geometry matplotlib seaborn pandas einops

import transformers, torch
print(f"transformers {transformers.__version__}")
print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("\nDone!")

In [ ]:
# Configuration

import sys, os
sys.path.insert(0, '.')
PHASE = 'full'

OUTPUT_BASE = './results/saprot_scaling_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

CONFIG = {
    'quick': {
        'n_sequences': 500,
        'seq_length': 200,
        'models': [
            'westlake-repl/SaProt_35M_AF2',    # 35M
            'westlake-repl/SaProt_650M_AF2',   # 650M
        ],
        'batch_size': 8,
        'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10],
    },
    'full': {
        'n_sequences': 10000,
        'seq_length': 200,
        'models': [
            'westlake-repl/SaProt_35M_AF2',              # 35M
            'westlake-repl/SaProt_650M_AF2',             # 650M
            'westlake-repl/SaProt_1.3B_AFDB_OMG_NCBI',  # 1.3B
        ],
        'batch_size': 8,
        'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10],
        # Batch overrides for large models
        'batch_overrides': {
            'westlake-repl/SaProt_1.3B_AFDB_OMG_NCBI': 2,
        },
    },
}

config = CONFIG[PHASE]
print(f"Phase: {PHASE.upper()}")
print(f"Architecture: SaProt (structure-aware ESM)")
print(f"Sequences: {config['n_sequences']}")
print(f"Seq length: {config['seq_length']} amino acids")
print(f"Models: {len(config['models'])} (35M -> 1.3B scaling)")

In [ ]:
# Generate Synthetic Protein Sequences

import numpy as np

AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

def generate_protein_sequences(n_sequences, seq_length, seed=320):
    rng = np.random.default_rng(seed)
    sequences = [
        ''.join(rng.choice(AMINO_ACIDS, size=seq_length))
        for _ in range(n_sequences)
    ]
    print(f"Generated {len(sequences)} protein sequences (length={seq_length})")
    print(f"Sample: {sequences[0][:50]}...")
    return sequences

sequences = generate_protein_sequences(config['n_sequences'], config['seq_length'])

In [ ]:
# Protein Perturbation Suite

from dataclasses import dataclass, field

@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: list
    params: dict = field(default_factory=dict)
    description: str = ''

def mutate_protein(sequence, mutation_rate, rng):
    seq = list(sequence)
    n_mutations = max(1, int(len(seq) * mutation_rate))
    positions = rng.choice(len(seq), size=n_mutations, replace=False)
    for pos in positions:
        original = seq[pos]
        alternatives = [aa for aa in AMINO_ACIDS if aa != original]
        seq[pos] = rng.choice(alternatives)
    return ''.join(seq)

def reverse_sequence(sequence):
    return sequence[::-1]

class ProteinPerturbationSuite:
    def __init__(self, seed=320, snp_rates=None, include_reverse=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_reverse = include_reverse

    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f"aa_sub_{int(rate * 100)}pct"
            perturbed = [mutate_protein(s, rate, self.rng) for s in sequences]
            results[name] = PerturbedSet(
                name=name, category='substitution', sequences=perturbed,
                params={'mutation_rate': rate},
                description=f'Random AA substitutions at {rate*100:.0f}% rate',
            )
        if self.include_reverse:
            results['reverse'] = PerturbedSet(
                name='reverse', category='reverse',
                sequences=[reverse_sequence(s) for s in sequences],
                params={}, description='Full sequence reversal',
            )
        return results

suite = ProteinPerturbationSuite(seed=320, snp_rates=config['snp_rates'])
test_perturbed = suite.run_all(sequences[:5])
for name, pset in test_perturbed.items():
    dists = [sum(a != b for a, b in zip(o, p)) / len(o)
             for o, p in zip(sequences[:5], pset.sequences)]
    print(f"  {name}: mean_hamming={np.mean(dists):.4f}")
print("Perturbation suite ready")

In [ ]:
# SaProt Model Wrapper
#
# SaProt uses a structure-aware vocabulary: each residue is paired with
# a Foldseek 3Di structure token. For sequence-only input (no structure),
# we use '#' as the structure token for every position.
# Example: "MEV" -> "M#E#V#"

import torch
import gc
import numpy as np
from transformers import EsmTokenizer, EsmForMaskedLM


def aa_to_saprot_format(seq):
    """Convert plain AA sequence to SaProt format with '#' structure tokens."""
    return "".join(aa + "#" for aa in seq)


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  GPU memory after cleanup: {mem:.2f} GB")


def make_saprot_embedding_fn(model_name, batch_size=8):
    """Load SaProt and return embedding function."""
    print(f"Loading {model_name}...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  Device: {device}")

    tokenizer = EsmTokenizer.from_pretrained(model_name)
    model = EsmForMaskedLM.from_pretrained(model_name).to(device).eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB")
    else:
        print(f"  Params: {n_params:.1f}M")

    max_len = getattr(model.config, 'max_position_embeddings', 1026)
    print(f"  Max position embeddings: {max_len}")

    # Verify tokenizer works with SaProt format
    test_sa = "M#E#V#"
    test_tokens = tokenizer.tokenize(test_sa)
    print(f"  Tokenizer check: '{test_sa}' -> {test_tokens}")

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size

        for i in range(0, len(sequences), batch_size):
            batch_seqs = sequences[i:i + batch_size]
            batch_idx = i // batch_size + 1

            if batch_idx % 20 == 0 or batch_idx == n_batches:
                print(f"    Batch {batch_idx}/{n_batches}", end='\r')

            # Convert plain AA sequences to SaProt format
            sa_seqs = [aa_to_saprot_format(s) for s in batch_seqs]

            tokens = tokenizer(
                sa_seqs, return_tensors='pt', padding=True,
                truncation=True, max_length=max_len,
            )
            tokens = {key: v.to(device) for key, v in tokens.items()}

            outputs = model(**tokens, output_hidden_states=True)
            last_hidden = outputs.hidden_states[-1]

            attention_mask = tokens['attention_mask'].unsqueeze(-1)
            pooled = (last_hidden * attention_mask).sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1)
            embeddings.append(pooled.cpu().numpy())

        print()
        return np.concatenate(embeddings, axis=0)

    return embed, model, tokenizer, n_params

print("SaProt wrapper ready")

In [ ]:
# Evaluation Harness

from evaluation_harness import StabilityHarness

harness = StabilityHarness(
    window_size=0, metric='cosine', n_splits=30,
    seed=320, max_samples=2500, n_bootstrap=config['n_bootstrap'],
)
print(f"Harness configured (bootstrap={config['n_bootstrap']})")

In [ ]:
# Run Experiment

import os, time, pandas as pd
from evaluation_harness import ModelReport

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print('=' * 70)
print(f'SAPROT SCALING EXPERIMENT -- Phase: {PHASE.upper()}')
print('=' * 70)

reports = []
model_param_counts = []
all_detailed_rows = []
batch_overrides = config.get('batch_overrides', {})

for model_idx, model_name in enumerate(config['models']):
    print(f"\n{'='*70}")
    print(f"[{model_idx+1}/{len(config['models'])}] {model_name}")
    print('=' * 70)

    start_time = time.time()
    bs = batch_overrides.get(model_name, config['batch_size'])
    embed_fn, model_obj, tokenizer_obj, n_params_m = make_saprot_embedding_fn(
        model_name, bs)
    model_param_counts.append(n_params_m)

    print('\nGenerating perturbations...')
    perturbed_sets = suite.run_all(sequences)

    safe_name = model_name.replace('/', '_').replace('-', '_')
    cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'

    if os.path.exists(cache_clean):
        print(f'Loading cached: {cache_clean}')
        embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings ({len(sequences)} sequences)...')
        embeddings_clean = embed_fn(sequences)
        np.save(cache_clean, embeddings_clean)
    print(f'  Shape: {embeddings_clean.shape}')

    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
        if os.path.exists(cache_pert):
            perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])

    del model_obj, tokenizer_obj
    cleanup_gpu()

    print('\nRunning Shesha evaluation...')
    report = harness.evaluate_all_perturbations(
        model_name=model_name, embeddings_clean=embeddings_clean,
        perturbed_dict=perturbed_embeddings)
    reports.append(report)

    short_name = model_name.split('/')[-1]
    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({
            'model': short_name, 'size_M': round(n_params_m, 1),
            'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0),
            'rdm_drift': metrics.get('rdm_drift', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0),
            'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0),
        })

    elapsed = time.time() - start_time
    summary = report.summary()
    print(f'\nCompleted in {elapsed/60:.1f} min')
    print(f'  Composite: {summary["mean_composite_stability"]:.4f}')

df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/saprot_scaling_{PHASE}_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nSaved to {detailed_path}')
print('\n' + '=' * 70)
print('EXPERIMENT COMPLETE')
print('=' * 70)

In [ ]:
# Visualization -- Scaling Plot

import matplotlib.pyplot as plt

from evaluation_harness import compare_models
comparison = compare_models(reports, output_dir=RESULTS_DIR)

summaries = [r.summary() for r in reports]
sizes = model_param_counts
composites = [s['mean_composite_stability'] for s in summaries]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sizes, composites, 'g^-', linewidth=2, markersize=14, label='SaProt')
for i, (sz, comp) in enumerate(zip(sizes, composites)):
    ax.annotate(f'{comp:.4f}', (sz, comp), textcoords='offset points',
                xytext=(0, 12), ha='center', fontsize=10)

ax.set_xscale('log')
ax.set_xlabel('Parameters (M)', fontsize=12)
ax.set_ylabel('Composite Stability', fontsize=12)
ax.set_title('SaProt: Geometric Stability vs Model Size', fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/saprot_scaling_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cross-Architecture Comparison (SaProt vs ESM-2)

import matplotlib.pyplot as plt, pandas as pd, glob

fig, ax = plt.subplots(figsize=(12, 7))

# SaProt (this experiment)
ax.plot(sizes, composites, 'g^-', linewidth=2, markersize=14, label='SaProt (struct-aware)')

# Overlay ESM-2 and other protein models
base = f'./results'
for label, pattern, color, marker in [
    ('ESM-2 (seq-only)', f'{base}/esm2_scaling_*/results/*_detailed.csv', 'tab:red', 'o-'),
    ('ProtMamba (SSM)', f'{base}/protmamba_scaling_*/results/*_detailed.csv', 'tab:purple', '*-'),
    ('Caduceus (DNA SSM)', f'{base}/caduceus_scaling_*/results/*_detailed.csv', 'tab:blue', 'D-'),
    ('NT (DNA Transformer)', f'{base}/nt_scaling_*/results/*_detailed.csv', 'tab:orange', 's-'),
]:
    files = glob.glob(pattern)
    if files:
        df = pd.read_csv(files[0])
        agg = df.groupby('size_M')['composite'].mean().reset_index().sort_values('size_M')
        ax.plot(agg['size_M'], agg['composite'], marker,
                color=color, linewidth=2, markersize=10, label=label)

ax.set_xscale('log')
ax.set_xlabel('Parameters (M)', fontsize=12)
ax.set_ylabel('Composite Stability', fontsize=12)
ax.set_title('SaProt vs ESM-2: Structure-Awareness Effect on Stability', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/saprot_crossexp_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()